# Практическая работа №4. Часть 1З
## Звук как сигнал: спектрограмма, мел-шкала, распознавание голосовых команд

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ExPl0i/AI-10-Speech-Signal/blob/main/%D0%9F%D0%A0-%D0%97%D0%B2%D1%83%D0%BA%20%D0%BA%D0%B0%D0%BA%20%D1%81%D0%B8%D0%B3%D0%BD%D0%B0%D0%BB.ipynb)

**Цель:** превратить звук в картинку, которую понимает сверточная сеть, и обучить распознаватель шести голосовых команд водителя беспилотного автомобиля: «вперед», «стоп», «налево», «направо», «быстрее», «медленнее». Это первая часть голосового помощника, который мы соберем на занятии Т2.Б2.5.

> **Правило части.** Спектр, спектрограмму, мел-фильтры, логарифмическую мел-спектрограмму, выравнивание длины, разбиение по дикторам и аугментацию пишем сами на NumPy и PyTorch. Библиотека `torchaudio` нужна только для сверки.

**Данные.** Есть два источника, выбор – в первой ячейке (`DATA_SOURCE`).
- `'группа'` – ваши записи из домашнего задания Т2.Б1.5. Преподаватель переводит их скриптом `prepare_group.py` в одинаковый формат (WAV, 16 000 отсчетов в секунду, моно, 16 бит) и раскладывает в папку `data/commands/<ученик>/<команда>_<номер>.wav`. Записи голоса – персональные данные: они лежат только на школьном носителе и не выкладываются в открытый доступ.
- `'запасной'` – подборка из открытого набора Google Speech Commands v2 (лицензия CC BY 4.0, 2018): английские слова forward, stop, left, right, up, down играют роли команд «вперед», «стоп», «налево», «направо», «быстрее», «медленнее». Каждый файл – одна секунда, у каждого диктора свой код в начале имени файла. Подборку делает `prepare_data.py` или первая ячейка раздела З4 (скачивает архив набора, около 2,3 ГБ, и оставляет нужные файлы).

**Два режима** выбираются в первой ячейке; оба работают на процессоре.
- `'легкий'` – компьютер в аудитории: запасной набор по 200 записей на команду, сеть SimpleCNN с 8 и 16 каналами, 15 эпох.
- `'полный'` – Colab или мощный компьютер: по 800 записей на команду, сеть с 16 и 32 каналами, 25 эпох.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка в комментарии `# <--`. Проверки на маленьких примерах и синтезированных сигналах должны совпасть с тем, что стоит после слова «ожидается», точно. Результаты обучения зависят от данных и случайных чисел, поэтому для них ожидания даны с запасом.

## Как звук становится картинкой

**1. Звук как колебание давления.** Звук – это колебание давления воздуха. Микрофон превращает давление в напряжение, а звуковая карта измеряет напряжение через равные промежутки времени. Число измерений (отсчетов) в секунду – **частота дискретизации** $f_s$. Для речи обычно берут $f_s = 16\,000$ Гц, для музыки – 44 100 Гц.

**Теорема Котельникова** (в англоязычной литературе ее называют теоремой Найквиста – Шеннона): сигнал без частот выше $f_{max}$ восстанавливается по отсчетам без потерь, если $f_s > 2 f_{max}$. Половина частоты дискретизации $f_s / 2$ называется частотой Найквиста: при $f_s = 16\,000$ Гц это 8000 Гц, и этого хватает для речи. Если в сигнале есть частота выше $f_s / 2$, она не пропадает, а «отражается» и выглядит как другая частота: тон 9000 Гц при $f_s = 16\,000$ Гц неотличим от тона $16\,000 - 9000 = 7000$ Гц. Это явление называют **наложением частот** (англ. aliasing), поэтому перед дискретизацией высокие частоты отрезают фильтром.

**Глубина квантования** – сколько битов на один отсчет. При $b$ битах отсчет принимает $2^b$ значений, и округление добавляет шум. Отношение сигнал/шум для синусоиды на всю шкалу примерно $6{,}02 b + 1{,}76$ дБ: 50 дБ при 8 битах, 98 дБ при 16 битах. Децибелы считают так: $\mathrm{SNR} = 10 \lg (P_{сигнала} / P_{шума})$, где $P$ – средний квадрат отсчетов (мощность). 10 дБ – мощность в 10 раз больше, 20 дБ – в 100 раз.

**Формат WAV** хранит отсчеты как есть, без сжатия, после заголовка в 44 байта. Одна секунда речи при 16 000 Гц, 16 битах и одном канале – $16\,000 \cdot 2 + 44 = 32\,044$ байта, минута – около 1,9 МБ. Форматы MP3, AAC (файлы .m4a с телефона) и Opus сжимают звук с потерями, поэтому записи с телефона сначала переводят в WAV.

**2. Разложение на частоты.** Любой сигнал длины $N$ можно представить суммой синусоид. **Дискретное преобразование Фурье** (ДПФ) находит, сколько в сигнале каждой частоты:
$$X_k = \sum_{n=0}^{N-1} x_n e^{-2\pi i k n / N}, \qquad k = 0, 1, \dots, N - 1.$$
Номер $k$ соответствует частоте $k \cdot f_s / N$ Гц, нужны номера от 0 до $N/2$ (остальные для вещественного сигнала повторяют их). Амплитуда синусоиды частоты $k f_s / N$ равна $2|X_k| / N$. Прямой расчет по формуле требует $N^2$ операций, а **быстрое преобразование Фурье** (БПФ, алгоритм Кули – Тьюки) требует около $N \log_2 N$: для секунды звука ($N = 16\,000$) это в тысячу раз быстрее. В NumPy – `np.fft.rfft` (только номера от 0 до $N/2$) и `np.fft.rfftfreq` (их частоты).

**3. Спектрограмма.** Речь меняется во времени: в слове «стоп» сначала шипящий звук «с» с высокими частотами, потом гласный «о» с низкими. Поэтому сигнал режут на короткие перекрывающиеся **кадры**, каждый умножают на **оконную функцию** и для каждого кадра считают спектр. Получается картинка: по горизонтали время, по вертикали частота, яркость – мощность $|X_k|^2$. Стандарт для речи: окно 25 мс (400 отсчетов при 16 000 Гц), шаг 10 мс (160 отсчетов). Секунда звука дает $1 + 16\,000 / 160 = 101$ кадр, каждый кадр – $400 / 2 + 1 = 201$ частота с шагом $16\,000 / 400 = 40$ Гц.

Окно Ханна $w_n = 0{,}5 - 0{,}5 \cos(2\pi n / N)$ плавно гасит края кадра. Без него обрыв сигнала на краях дает в спектре ложные частоты (растекание спектра). Чтобы кадры покрыли и края сигнала, сигнал дополняют с обеих сторон на половину окна его зеркальным отражением.

**Компромисс между временем и частотой.** Длинное окно дает точные частоты (шаг $f_s / N$ мельче), но размывает быстрые события; короткое окно точно показывает момент, но частоты в нем грубые. Окно 25 мс – компромисс для речи: звук гласной почти не меняется за это время, а частотный шаг 40 Гц достаточен.

**4. Мел-шкала.** Человек слышит высоту звука нелинейно: разницу между 100 и 200 Гц мы слышим так же отчетливо, как между 1000 и 1200 Гц, а между 7000 и 7100 Гц почти не слышим. **Мел-шкала** выравнивает высоту по восприятию:
$$m = 2595 \lg\left(1 + \frac{f}{700}\right), \qquad f = 700\left(10^{m / 2595} - 1\right).$$
1000 Гц – это 1000 мел, 8000 Гц – 2840 мел. **Мел-фильтры** – треугольники, центры которых стоят равномерно по мел-шкале: на низких частотах узкие (ширина первого около 90 Гц), на высоких широкие (ширина последнего около 1000 Гц). Каждый фильтр складывает мощность спектрограммы со своими весами, и вместо 201 частоты остается, например, 40 мел-полос. **Логарифмическая мел-спектрограмма** – мощности мел-полос в децибелах: $10 \lg(\max(P, 10^{-10}))$. Логарифм тоже следует слуху: громкость мы воспринимаем по отношению мощностей, а не по разности. Это стандартный вход моделей речи, в том числе Whisper (80 мел-полос, окно 25 мс, шаг 10 мс).

**5. Мел-кепстральные коэффициенты** (MFCC) одной строкой: дискретное косинусное преобразование логарифмической мел-спектрограммы каждого кадра, из которого оставляют первые 13–20 чисел; раньше это был главный признак распознавания речи, а нейросетям обычно дают саму мел-спектрограмму.

**6. Аугментация звука** (связь с Б2.4 Темы 1). Записей мало, а дикторы, микрофоны и шум в машине разные. Поэтому при обучении каждую запись случайно меняют, не меняя смысла:
- **шум**: к записи прибавляют шум (белый, розовый, шум двигателя) с заданным отношением сигнал/шум, например от 5 до 20 дБ. Шум масштабируют множителем $k = \sqrt{P_x / (P_n \cdot 10^{\mathrm{SNR}/10})}$;
- **сдвиг во времени**: команду сдвигают на долю секунды, освободившееся место заполняют нулями;
- **изменение скорости**: запись проигрывают быстрее или медленнее (в 0,9–1,1 раза), при этом меняется и высота голоса. Ускорение в 1,25 раза превращает тон 440 Гц в 550 Гц;
- **закрытие полос спектрограммы** (SpecAugment, Парк и соавторы, 2019): на спектрограмме закрывают случайную полосу частот и случайный отрезок времени. Сеть учится узнавать команду, даже если часть информации пропала.

**7. Разбиение по дикторам.** У каждого человека свой голос, поэтому записи одного диктора похожи друг на друга сильнее, чем записи одной команды разными людьми. Если записи одного диктора попадут и в обучение, и в проверку, сеть получит высокую точность, «узнавая голос», а на новом водителе ошибется. Поэтому дикторов делят на обучающих и проверочных целиком: все записи диктора идут в одну часть.

**Нужное из NumPy, PyTorch и torchaudio.**
- `np.fft.rfft(x)`, `np.fft.rfftfreq(n, d=1 / sr)`; `np.round`, `np.pad(x, (слева, справа))`, `np.interp(новые_точки, старые_точки, значения)`, `np.cumsum`;
- `F.pad(x, (p, p), mode='reflect')` – зеркальное дополнение (для входа размера (пакет, 1, длина)); `x.unfold(-1, n_fft, hop)` – кадры с шагом; `torch.hann_window(n_fft)` – окно Ханна; `torch.fft.rfft(frames, dim=-1)`; `torch.log10`, `torch.clamp(x, min=...)`;
- `torchaudio.transforms.Spectrogram`, `torchaudio.functional.melscale_fbanks`, `torchaudio.transforms.MelSpectrogram`, `torchaudio.transforms.AmplitudeToDB` – библиотечные версии для сверки;
- `IPython.display.Audio(x, rate=16000)` – послушать сигнал прямо в ноутбуке.

In [ ]:
# Ячейка подготовки: режим, источник данных, общие функции
MODE = 'легкий'          # 'легкий' – компьютер в аудитории; 'полный' – Colab или мощный компьютер
DATA_SOURCE = 'запасной' # 'группа' – записи группы в data/commands; 'запасной' – подборка Speech Commands

import platform
import sys
import tarfile
import time
import urllib.request
import wave
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display
import matplotlib.pyplot as plt
from IPython.display import Audio, Image, display

assert MODE in ('легкий', 'полный') and DATA_SOURCE in ('группа', 'запасной')
try:
    import torchaudio
    HAVE_TA = True
except ImportError:
    HAVE_TA = False
print('Python', platform.python_version(), '| PyTorch', torch.__version__, '| torchaudio', torchaudio.__version__ if HAVE_TA else 'нет (сверки будут пропущены)',
      '| режим:', MODE, '| данные:', DATA_SOURCE)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
SR, N_FFT, HOP, N_MELS, CLIP = 16000, 400, 160, 40, 16000   # частота, окно 25 мс, шаг 10 мс, мел-полосы, длина записи 1 с
SETTINGS = {'легкий': dict(per_class=200, width=8, epochs=15), 'полный': dict(per_class=800, width=16, epochs=25)}
S = SETTINGS[MODE]
COMMANDS = ['вперед', 'стоп', 'налево', 'направо', 'быстрее', 'медленнее']
RESERVE_WORDS = {'вперед': 'forward', 'стоп': 'stop', 'налево': 'left', 'направо': 'right', 'быстрее': 'up', 'медленнее': 'down'}
NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'

def expect(light, full):
    """Ожидаемое значение для выбранного режима."""
    return light if MODE == 'легкий' else full

def show_fig(fig, name):
    fig.tight_layout(); fig.savefig(name, dpi=110); plt.close(fig)
    display(Image(name))

def read_wav(path):
    """WAV в массив float32 от −1 до 1, моно, 16 000 Гц."""
    with wave.open(str(path), 'rb') as f:
        sr, ch, width, n = f.getframerate(), f.getnchannels(), f.getsampwidth(), f.getnframes()
        raw = f.readframes(n)
    dtype = {1: np.uint8, 2: np.int16, 4: np.int32}[width]
    x = np.frombuffer(raw, dtype=dtype).astype(np.float32)
    x = (x - 128) / 128 if width == 1 else x / float(2 ** (8 * width - 1))
    x = x.reshape(-1, ch).mean(axis=1)
    if sr != SR:   # простое изменение частоты дискретизации линейной интерполяцией
        x = np.interp(np.arange(int(len(x) * SR / sr)) * sr / SR, np.arange(len(x)), x).astype(np.float32)
    return x

### Задание З1. Тон, смесь тонов и спектр

Допишите три функции по пп. 1–2 теории:
- `tone(freq, dur, sr, amp)` – синусоида $a \sin(2\pi f t)$ в моменты $t = n / f_s$, $n = 0, 1, \dots$;
- `spectrum(x, sr)` – частоты `np.fft.rfftfreq` и амплитуды $2|X_k| / N$ через `np.fft.rfft`;
- `quantize(x, bits)` – округление к ближайшему из уровней с шагом $1 / (2^{b-1} - 1)$: умножить на $2^{b-1} - 1$, округлить, разделить обратно.

Затем найдите два главных пика смеси тонов 440 Гц и 1000 Гц, проверьте наложение частот на тоне 9000 Гц и измерьте шум квантования.

In [ ]:
def tone(freq, dur=1.0, sr=SR, amp=1.0):
    t = ...                                         # <-- моменты отсчетов в секундах
    return ...

def spectrum(x, sr=SR):
    X = ...
    freqs = ...                                     # <-- частоты номеров 0 ... N/2
    return freqs, ...                               # <-- амплитуды

def quantize(x, bits):
    q = 2 ** (bits - 1) - 1
    return ...

def snr_db(clean, noisy):
    """Отношение сигнал/шум в децибелах."""
    return 10 * np.log10(np.mean(clean ** 2) / np.mean((noisy - clean) ** 2))


mix = tone(440, amp=0.5) + tone(1000, amp=0.3)
freqs, amps = spectrum(mix)
top = np.argsort(-amps)[:2]
print('пики:', freqs[top].tolist(), amps[top].round(3).tolist(), '  ожидается [440.0, 1000.0] [0.5, 0.3]')
f9, a9 = spectrum(tone(9000))
print('тон 9000 Гц при 16000 Гц виден на', f9[a9.argmax()], 'Гц', '  ожидается 7000.0 – наложение частот')
s = tone(440)
for b in (16, 8, 4):
    print(f'{b:2d} бит: сигнал/шум {snr_db(s, quantize(s, b)):.1f} дБ, по формуле {6.02 * b + 1.76:.1f}')
print('  ожидается: 97.9, 50.4, 25.1 – близко к формуле')

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(np.arange(320) / SR * 1000, mix[:320], color=DEEP); ax[0].set_xlabel('время, мс'); ax[0].set_title('смесь тонов, первые 20 мс')
ax[1].plot(freqs, amps, color=DEEP); ax[1].set_xlim(0, 2000); ax[1].set_xlabel('частота, Гц'); ax[1].set_title('спектр')
show_fig(fig, 'z1_spectrum.png')
display(Audio(mix, rate=SR))
display(Audio(quantize(s, 4) * 0.3, rate=SR))   # тон 440 Гц при 4 битах: слышен шум квантования

**Вопрос З1.** а) Почему для речи хватает частоты дискретизации 16 000 Гц, а для музыки берут 44 100 Гц? б) Что будет, если записать свист на частоте 9 кГц с частотой дискретизации 16 кГц без фильтра? в) Сколько места занимает минута записи в WAV при 16 000 Гц, 16 битах и одном канале и почему записи с телефона весят меньше?

*Ваш ответ:*

### Задание З2. Своя спектрограмма

Допишите по п. 3 теории:
- `frame_signal(x, n_fft, hop)` – зеркальное дополнение на `n_fft // 2` отсчетов с каждой стороны (`F.pad(..., mode='reflect')` работает с входом размера (пакет, 1, длина)) и нарезка на кадры `unfold(-1, n_fft, hop)`; функция должна принимать и один сигнал размера (длина,), и пакет (пакет, длина);
- `my_spectrogram(x, n_fft, hop)` – кадры, умноженные на окно Ханна `torch.hann_window(n_fft)`, преобразование Фурье `torch.fft.rfft` по последней оси, мощность $|X|^2$ и перестановка осей, чтобы частота шла перед временем: результат (…, частоты, кадры).

In [ ]:
def frame_signal(x, n_fft=N_FFT, hop=HOP):
    shape = x.shape[:-1]
    x3 = x.reshape(-1, 1, x.shape[-1])                       # (пакет, 1, длина)
    pad = n_fft // 2
    xp = ...                                        # <-- зеркальное дополнение с двух сторон
    frames = ...                                    # <-- (пакет, кадры, n_fft)
    return frames.reshape(*shape, *frames.shape[-2:])

def my_spectrogram(x, n_fft=N_FFT, hop=HOP):
    frames = frame_signal(x, n_fft, hop) * ...      # <-- окно Ханна
    power = ...                                     # <-- мощность |X|^2
    return ...                                      # <-- (…, частоты, кадры)


x1000 = torch.tensor(tone(1000), dtype=torch.float32)
spec = my_spectrogram(x1000)
print(tuple(spec.shape), '  ожидается (201, 101): 201 частота с шагом 40 Гц, 101 кадр')
print('самая мощная частота:', int(spec.mean(dim=1).argmax()) * SR // N_FFT, 'Гц', '  ожидается 1000')
print('пакет:', tuple(my_spectrogram(torch.stack([x1000, x1000])).shape), '  ожидается (2, 201, 101)')
test = torch.tensor(tone(440, amp=0.5) + tone(1000, amp=0.3) + 0.01 * np.random.default_rng(0).normal(size=SR), dtype=torch.float32)
if HAVE_TA:
    ref = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=HOP, power=2.0)(test)
    err = ((my_spectrogram(test) - ref).abs().max() / ref.abs().max()).item()
    print(f'совпадает с torchaudio: {err < 1e-4} (относительная ошибка {err:.1e})', '  ожидается True')

In [ ]:
# Готовая ячейка: спектрограмма свиста с растущей частотой и компромисс «время – частота»
t = np.arange(SR) / SR
chirp = 0.5 * np.sin(2 * np.pi * (200 * t + 3400 * t ** 2))         # частота растет от 200 до 7000 Гц
two = tone(1000, amp=0.3) + tone(1080, amp=0.3)                      # два близких тона
two[8000:8040] += 1.0                                                # и короткий щелчок посередине
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
db = lambda p: 10 * torch.log10(torch.clamp(p, min=1e-10))
ax[0].imshow(db(my_spectrogram(torch.tensor(chirp, dtype=torch.float32))), origin='lower', aspect='auto', cmap='Blues', extent=[0, 1, 0, SR / 2])
ax[0].set_title('свист 200 → 7000 Гц'); ax[0].set_xlabel('время, с'); ax[0].set_ylabel('частота, Гц')
for k, n in enumerate((128, 2048)):
    p = my_spectrogram(torch.tensor(two, dtype=torch.float32), n_fft=n, hop=n // 4)
    ax[k + 1].imshow(db(p), origin='lower', aspect='auto', cmap='Blues', extent=[0, 1, 0, SR / 2])
    ax[k + 1].set_ylim(0, 3000); ax[k + 1].set_title(f'1000 и 1080 Гц + щелчок, окно {n * 1000 // SR} мс'); ax[k + 1].set_xlabel('время, с')
show_fig(fig, 'z2_spectrogram.png')
display(Audio(chirp, rate=SR))
print('ожидается: наклонная линия у свиста; при окне 8 мс два тона слились в одну полосу, но щелчок – тонкая вертикальная линия;',
      'при окне 128 мс тона видны раздельно, а щелчок размазан во времени')

**Вопрос З2.** а) Почему для речи выбирают окно 25 мс, а не 8 мс и не 128 мс? Опирайтесь на картинку. б) Зачем умножать кадр на окно Ханна? в) Сколько кадров даст запись длиной 2 секунды при шаге 10 мс и почему кадров на один больше, чем «длина / шаг»?

*Ваш ответ:*

### Задание З3. Мел-фильтры и логарифмическая мел-спектрограмма

Допишите по п. 4 теории:
- `hz_to_mel(f)` и `mel_to_hz(m)` – формулы мел-шкалы; чтобы функции работали и с числами, и с тензорами, вход переводится в тензор `torch.as_tensor(f, dtype=torch.float64)`, логарифм – `torch.log10`;
- `mel_filterbank(n_freqs, sr, n_mels, f_min, f_max)` – матрица (частоты, мел-полосы): частоты спектра `torch.linspace(0, sr // 2, n_freqs)`, точки `n_mels + 2`, равномерные по мел-шкале от $f_{min}$ до $f_{max}$, переведенные в герцы; фильтр $j$ – треугольник с вершиной в точке $j + 1$ и основанием от точки $j$ до точки $j + 2$: вес частоты $f$ равен $\max\big(0, \min\big(\frac{f - p_j}{p_{j+1} - p_j}, \frac{p_{j+2} - f}{p_{j+2} - p_{j+1}}\big)\big)$;
- `log_mel(x, fb)` – спектрограмма, свертка с фильтрами (`fb.T @ spec`) и децибелы $10 \lg(\max(P, 10^{-10}))$.

In [ ]:
def hz_to_mel(f):
    return ...

def mel_to_hz(m):
    return ...

def mel_filterbank(n_freqs=N_FFT // 2 + 1, sr=SR, n_mels=N_MELS, f_min=0.0, f_max=SR / 2):
    freqs = torch.linspace(0, sr // 2, n_freqs, dtype=torch.float64)
    pts = ...                                       # <-- n_mels + 2 точек, равномерных по мелам
    left, center, right = pts[:-2], pts[1:-1], pts[2:]                 # для каждого фильтра
    up = ...                                        # <-- левый склон треугольника
    down = ...                                      # <-- правый склон
    return ....float()                              # <-- (частоты, мел-полосы)

FB = mel_filterbank()

def log_mel(x, fb=FB):
    x = torch.as_tensor(x, dtype=torch.float32)
    mel =...                                        # <-- (…, мел-полосы, кадры)
    return ...


print(round(float(hz_to_mel(1000)), 2), round(float(hz_to_mel(8000)), 2), round(float(mel_to_hz(hz_to_mel(440))), 2), '  ожидается 999.99 2840.02 440.0')
pts = mel_to_hz(torch.linspace(0, float(hz_to_mel(8000)), N_MELS + 2))
print('центры первых фильтров:', pts[1:4].numpy().round(1).tolist(), 'Гц, ширина первого', round(float(pts[2] - pts[0]), 1), 'последнего', round(float(pts[-1] - pts[-3]), 1),
      '  ожидается [44.4, 91.6, 141.7] 91.6 1006.3')
print(tuple(FB.shape), 'пустых фильтров:', int((FB.sum(0) == 0).sum()), '  ожидается (201, 40) 0')
lm = log_mel(test)
print(tuple(lm.shape), '  ожидается (40, 101)')
if HAVE_TA:
    fb_ref = torchaudio.functional.melscale_fbanks(n_freqs=N_FFT // 2 + 1, f_min=0.0, f_max=SR / 2, n_mels=N_MELS, sample_rate=SR, norm=None, mel_scale='htk')
    mel_ref = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)(test)
    db_ref = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=None)(mel_ref)
    print('фильтры совпадают с torchaudio:', torch.allclose(FB, fb_ref, atol=1e-5),
          '| мел-спектрограмма в дБ:', (lm - db_ref).abs().max().item() < 0.01, '  ожидается True True')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(torch.linspace(0, SR // 2, N_FFT // 2 + 1), FB[:, ::2], color=DEEP, lw=1); ax[0].set_xlabel('частота, Гц'); ax[0].set_title('каждый второй из 40 мел-фильтров')
ax[1].imshow(log_mel(torch.tensor(chirp, dtype=torch.float32)), origin='lower', aspect='auto', cmap='Blues'); ax[1].set_title('свист: логарифмическая мел-спектрограмма')
ax[1].set_xlabel('кадр'); ax[1].set_ylabel('мел-полоса')
show_fig(fig, 'z3_mel.png')
print('ожидается: узкие треугольники слева, широкие справа; линия свиста на мел-спектрограмме изогнута – высокие частоты сжаты')

**Вопрос З3.** а) Почему мел-фильтры на высоких частотах шире, чем на низких? б) Во сколько раз мел-спектрограмма меньше обычной спектрограммы и почему это хорошо для маленькой сети? в) Зачем брать логарифм мощности?

*Ваш ответ:*

### Задание З4. Записи, выравнивание длины, разбиение по дикторам, аугментация

Готовая ячейка ниже загружает записи: записи группы из `data/commands` или запасную подборку Speech Commands из `data/reserve` (если ее нет, скачивает архив набора и оставляет по 800 записей на команду от разных дикторов, это несколько минут). Там же – генераторы шума: белый, розовый и синтезированный шум двигателя (гармоники частоты вспышек в цилиндрах и низкочастотный гул).

In [ ]:
# Готовая ячейка: загрузка записей и генераторы шума
DATA = Path('data')
SC_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'

def build_reserve(per_class=800, root=DATA / 'reserve'):
    """Подборка Speech Commands v2: по per_class записей на команду, по одной записи на диктора, и фоновые шумы."""
    want = {eng: rus for rus, eng in RESERVE_WORDS.items()}
    counts, speakers, n_noise = Counter(), defaultdict(set), 0
    print('скачиваю архив Speech Commands v2 (около 2,3 ГБ) и выбираю записи ...')
    with urllib.request.urlopen(SC_URL) as resp, tarfile.open(fileobj=resp, mode='r|gz') as tar:
        for m in tar:
            parts = m.name.split('/')
            if not m.isfile() or not m.name.endswith('.wav') or len(parts) < 2:
                continue
            word, fname = parts[-2], parts[-1]
            if word == '_background_noise_':
                dst = root / '_noise' / fname; n_noise += 1
            elif word in want and counts[word] < per_class and fname.split('_')[0] not in speakers[word]:
                speakers[word].add(fname.split('_')[0]); counts[word] += 1
                dst = root / want[word] / fname
            else:
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(tar.extractfile(m).read())
            if n_noise >= 6 and all(counts[w] >= per_class for w in want):
                break
    print('готово:', dict(counts), '| шумов:', n_noise)

def load_dataset():
    """Списки: сигналы, номера команд, дикторы."""
    waves, labels, speakers = [], [], []
    if DATA_SOURCE == 'группа':
        for path in sorted((DATA / 'commands').glob('*/*.wav')):
            cmd = path.stem.split('_')[0].lower().replace('ё', 'е')
            if cmd in COMMANDS:
                waves.append(read_wav(path)); labels.append(COMMANDS.index(cmd)); speakers.append(path.parent.name)
    else:
        root = DATA / 'reserve'
        if not all((root / c).exists() for c in COMMANDS):
            build_reserve()
        for k, cmd in enumerate(COMMANDS):
            for path in sorted((root / cmd).glob('*.wav'))[:S['per_class']]:
                waves.append(read_wav(path)); labels.append(k); speakers.append(path.stem.split('_')[0])
    assert waves, 'записей не найдено: проверьте папку data/commands или выберите DATA_SOURCE = "запасной"'
    return waves, np.array(labels), np.array(speakers)

def pink_noise(n, rng):
    spec = np.fft.rfft(rng.normal(size=n)) / np.sqrt(np.maximum(np.arange(n // 2 + 1), 1))
    x = np.fft.irfft(spec, n); return x / x.std()

def engine_noise(n, rng, rpm=2400):
    """Шум двигателя: 4 цилиндра, 4 такта – вспышки с частотой rpm / 60 · 2 Гц и ее гармоники, плюс низкочастотный гул."""
    t = np.arange(n) / SR; f0 = rpm / 60 * 2
    x = sum(np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k for k in range(1, 8))
    rumble = np.cumsum(rng.normal(size=n)); rumble -= np.convolve(rumble, np.ones(400) / 400, mode='same')
    x = x / x.std() + 0.7 * rumble / rumble.std(); return x / x.std()

t0 = time.time()
waves, labels, speakers = load_dataset()
rng_noise = np.random.default_rng(RANDOM_STATE)
NOISES = [rng_noise.normal(size=10 * SR), pink_noise(10 * SR, rng_noise), engine_noise(10 * SR, rng_noise)]
NOISES += [read_wav(p) for p in sorted((DATA / 'reserve' / '_noise').glob('*.wav'))]   # фоновые шумы Speech Commands, если они скачаны
lengths = np.array([len(w) for w in waves]) / SR
print(f'{len(waves)} записей за {time.time() - t0:.0f} с, дикторов {len(set(speakers))}, шумов {len(NOISES)}')
print('по командам:', {c: int((labels == k).sum()) for k, c in enumerate(COMMANDS)})
print(f'длина записей: от {lengths.min():.2f} до {lengths.max():.2f} с')
print('  ожидается для запасного набора:', expect('по 200', 'по 800'), 'записей на команду, длина до 1.00 с')
for k in (0, 1, 2):
    i = int(np.flatnonzero(labels == k)[0]); print(COMMANDS[k], speakers[i]); display(Audio(waves[i], rate=SR))

Допишите функции подготовки записей по пп. 6–7 теории:
- `fit_length(x, n)` – короткую запись дополнить нулями поровну слева и справа (слева $\lfloor (n - L)/2 \rfloor$); из длинной вырезать $n$ отсчетов вокруг «центра энергии» $c = \sum_i i\, x_i^2 / \sum_i x_i^2$: начало $\lfloor c - n/2 \rfloor$, но не меньше 0 и не больше $L - n$. Записи с телефона длиннее секунды и с тишиной по краям, а команда – там, где энергия;
- `split_by_speaker(speakers, test_share, seed)` – список дикторов по алфавиту, перемешать `np.random.default_rng(seed).permutation`, первые `round(test_share · число дикторов)` – проверочные; вернуть маску «запись обучающая»;
- `add_noise(x, noise, snr_db, rng)` – случайный кусок шума длины записи и множитель $k$ из п. 6 теории;
- `time_shift(x, shift)` – сдвиг на `shift` отсчетов вправо (отрицательный – влево) с заполнением нулями;
- `change_speed(x, factor)` – новая длина $\mathrm{round}(L / \text{factor})$, значения `np.interp` в точках $0, \text{factor}, 2\,\text{factor}, \dots$;
- `mask_bands(spec, f0, f_width, t0, t_width)` – копия спектрограммы, в которой полосы частот `f0 … f0 + f_width − 1` и кадры `t0 … t0 + t_width − 1` заменены нулями (работает и с NumPy, и с тензорами).

In [ ]:
def fit_length(x, n=CLIP):
    if len(x) < n:
        left = ...
        return np.pad(x, (left, n - len(x) - left))
    e = x.astype(np.float64) ** 2
    c = ...                                         # <-- центр энергии
    start = ...
    return x[start:start + n]

def split_by_speaker(speakers, test_share=0.2, seed=RANDOM_STATE):
    unique = np.array(sorted(set(speakers)))
    unique = unique[...]                            # <-- перемешать дикторов
    test = set(...)                                 # <-- первые – проверочные
    return np.array([s not in test for s in speakers])

def add_noise(x, noise, snr_db, rng):
    start = rng.integers(0, len(noise) - len(x) + 1)
    seg = noise[start:start + len(x)]
    k = ...                                         # <-- множитель шума
    return x + k * seg

def time_shift(x, shift):
    out = np.zeros_like(x)
    if shift >= 0:
        ...
    else:
        ...
    return out

def change_speed(x, factor):
    n = ...
    return ....astype(np.float32)

def mask_bands(spec, f0, f_width, t0, t_width):
    spec = spec.copy() if isinstance(spec, np.ndarray) else spec.clone()
    ...                                             # <-- полоса частот
    ...                                             # <-- отрезок времени
    return spec


burst = np.zeros(32000); burst[20000:21000] = 1.0
z = fit_length(burst, 16000)
print(len(z), int(np.argmax(z > 0)), z.sum(), '  ожидается 16000 7501 1000.0')
z = fit_length(np.ones(10000), 16000)
print(len(z), int(np.argmax(z > 0)), z.sum(), '  ожидается 16000 3000 10000.0')
m = split_by_speaker(['a', 'a', 'b', 'c', 'c', 'd', 'e', 'e', 'f', 'g'], test_share=0.3, seed=0)
print(m.astype(int).tolist(), '  ожидается [1, 1, 1, 0, 0, 1, 0, 0, 1, 1]')
x = tone(440, amp=0.5)
noisy = add_noise(x, np.random.default_rng(1).normal(size=48000), 10, np.random.default_rng(0))
print(f'сигнал/шум: {snr_db(x, noisy):.6f}', '  ожидается 10.000000')
print(time_shift(np.array([1, 2, 3, 4, 5]), 2).tolist(), time_shift(np.array([1, 2, 3, 4, 5]), -1).tolist(), '  ожидается [0, 0, 1, 2, 3] [2, 3, 4, 5, 0]')
fast = change_speed(tone(440), 1.25)
f, a = spectrum(fit_length(fast, 16000))
print(len(fast), len(change_speed(tone(440), 0.8)), f[a.argmax()], '  ожидается 12800 20000 550.0')
print(int((mask_bands(np.ones((40, 101)), 5, 8, 30, 20) == 0).sum()), '  ожидается 1448')

In [ ]:
# Готовая ячейка: одинаковая длина, разбиение по дикторам, примеры аугментации
X_wave = np.stack([fit_length(w, CLIP) for w in waves]).astype(np.float32)
train_mask = split_by_speaker(speakers, test_share=0.2)
print(f'обучение: {train_mask.sum()} записей, {len(set(speakers[train_mask]))} дикторов | проверка: {(~train_mask).sum()} записей, {len(set(speakers[~train_mask]))} дикторов')
print('общих дикторов:', len(set(speakers[train_mask]) & set(speakers[~train_mask])), '  ожидается 0')

def augment(x, rng):
    """Случайная аугментация одной записи: скорость, сдвиг, шум."""
    if rng.random() < 0.5:
        x = fit_length(change_speed(x, rng.uniform(0.9, 1.1)), CLIP)
    x = time_shift(x, int(rng.integers(-1600, 1601)))
    if rng.random() < 0.8:
        x = add_noise(x, NOISES[rng.integers(len(NOISES))], rng.uniform(5, 20), rng)
    return x.astype(np.float32)

i = int(np.flatnonzero(train_mask)[0])
rng = np.random.default_rng(0)
versions = {'исходная': X_wave[i],
            'шум двигателя, 5 дБ': add_noise(X_wave[i], NOISES[2], 5, rng),
            'быстрее в 1,1 раза': fit_length(change_speed(X_wave[i], 1.1), CLIP),
            'сдвиг на 0,1 с': time_shift(X_wave[i], 1600)}
fig, ax = plt.subplots(1, 5, figsize=(15, 2.8))
for k, (name, w) in enumerate(versions.items()):
    ax[k].imshow(log_mel(torch.tensor(w)), origin='lower', aspect='auto', cmap='Blues'); ax[k].set_title(name)
ax[4].imshow(mask_bands(log_mel(torch.tensor(X_wave[i])), 12, 6, 40, 15), origin='lower', aspect='auto', cmap='Blues'); ax[4].set_title('закрыты полосы')
show_fig(fig, 'z4_augment.png')
for name, w in versions.items():
    print(name); display(Audio(w, rate=SR))

**Вопрос З4.** а) Почему нельзя делить записи на обучающие и проверочные случайно, без учета дикторов? Что тогда покажет точность? б) Какая аугментация моделирует какую реальную ситуацию в машине? в) Почему изменение скорости меняет и высоту голоса и не испортит ли это команду?

*Ваш ответ:*

### Задание З5. Сверточная сеть на мел-спектрограммах

Мел-спектрограмма записи – картинка 40 × 101 с одним каналом, поэтому подходит сеть SimpleCNN из части 4А Темы 1: две свертки 3 × 3 с функцией ReLU и пулингом 2 × 2, затем полносвязный слой. После двух пулингов картинка 40 × 101 становится 10 × 25. Добавлен отсев (Б2.3 Темы 1) перед полносвязным слоем и параметр ширины `width` – число каналов первой свертки.

Допишите:
- в `SimpleCNN` – слои признаков, размер входа полносвязного слоя и проход вперед;
- в `train_epoch` – шаг обучения на пакете: оценки, перекрестная энтропия, обнуление градиентов, обратный проход, шаг оптимизатора.

Признаки нормируются средним и стандартным отклонением обучающей части. Готовая ячейка обучает две сети: без аугментации и с аугментацией (шум, сдвиг, скорость и закрытие полос) – и сравнивает их точность на дикторах, которых нет в обучении.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=len(COMMANDS), width=8, n_mels=N_MELS, n_frames=CLIP // HOP + 1, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            ...,                                    # <-- свертка 1 -> width, ReLU, пулинг
            ...,                                    # <-- свертка width -> 2·width, ReLU, пулинг
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(..., n_classes)  # <-- каналы · 10 · 25

    def forward(self, x):
        x = ...                                     # <-- признаки и вытягивание в вектор
        return ...

def train_epoch(model, X, y, opt, batch=64, gen=None):
    model.train()
    order = torch.randperm(len(X), generator=gen)
    total = 0.0
    for s in range(0, len(X), batch):
        idx = order[s:s + batch]
        logits = ...
        loss = ...
        ...
        ...                                         # <-- обратный проход
        ...
        total += loss.item() * len(idx)
    return total / len(X)

@torch.no_grad()
def predict(model, X, batch=256):
    model.eval()
    return torch.cat([model(X[s:s + batch]).argmax(1) for s in range(0, len(X), batch)])


net = SimpleCNN(width=8)
print(sum(p.numel() for p in net.parameters()), sum(p.numel() for p in SimpleCNN(width=16).parameters()), '  ожидается 25254 52806')
print(tuple(net.features(torch.zeros(2, 1, 40, 101)).shape), tuple(net(torch.zeros(2, 1, 40, 101)).shape), '  ожидается (2, 16, 10, 25) (2, 6)')

In [ ]:
# Готовая ячейка: признаки, обучение без аугментации и с ней
def features(wav_batch, chunk=512):
    """Логарифмические мел-спектрограммы пакета записей, по кускам, чтобы не тратить много памяти: (N, 1, 40, 101)."""
    wav_batch = np.asarray(wav_batch, dtype=np.float32)
    return torch.cat([log_mel(wav_batch[s:s + chunk]) for s in range(0, len(wav_batch), chunk)])[:, None]

F_all = features(X_wave)
y_all = torch.tensor(labels)
tr, te = torch.tensor(train_mask), torch.tensor(~train_mask)
MU, SD = F_all[tr].mean(), F_all[tr].std()
X_tr, X_te = (F_all[tr] - MU) / SD, (F_all[te] - MU) / SD
y_tr, y_te = y_all[tr], y_all[te]

def augmented_features(rng):
    """Новые случайные версии обучающих записей на каждую эпоху; закрытие полос – у половины."""
    Xa = (features([augment(w, rng) for w in X_wave[train_mask]]) - MU) / SD
    for k in range(len(Xa)):
        if rng.random() < 0.5:
            Xa[k] = mask_bands(Xa[k], int(rng.integers(0, 34)), int(rng.integers(1, 7)), int(rng.integers(0, 86)), int(rng.integers(1, 16)))
    return Xa

results, curves = {}, {}
for use_aug in (False, True):
    torch.manual_seed(RANDOM_STATE)
    model = SimpleCNN(width=S['width'])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    gen, rng = torch.Generator().manual_seed(RANDOM_STATE), np.random.default_rng(RANDOM_STATE)
    name = 'с аугментацией' if use_aug else 'без аугментации'
    curves[name] = []
    t0 = time.time()
    for epoch in range(S['epochs']):
        X_epoch = augmented_features(rng) if use_aug else X_tr
        loss = train_epoch(model, X_epoch, y_tr, opt, gen=gen)
        acc = (predict(model, X_te) == y_te).float().mean().item()
        curves[name].append(acc)
    results[name] = model
    train_acc = (predict(model, X_tr) == y_tr).float().mean().item()
    print(f'{name}: {time.time() - t0:.0f} с | точность на обучении {train_acc:.3f}, на новых дикторах {acc:.3f}')
print('ожидается: на новых дикторах намного выше случайного угадывания 1/6 ≈ 0.17, ориентир для запасного набора –',
      expect('выше 0.6', 'выше 0.8'), '; с аугментацией разрыв между обучением и проверкой меньше')

fig, ax = plt.subplots(figsize=(6, 3.2))
for (name, c), col in zip(curves.items(), (GRAY, DEEP)):
    ax.plot(range(1, len(c) + 1), c, color=col, label=name)
ax.set_xlabel('эпоха'); ax.set_ylabel('точность на новых дикторах'); ax.legend()
show_fig(fig, 'z5_curves.png')

**Вопрос З5.** а) Сравните точность на обучении и на новых дикторах для двух сетей. Где сильнее переобучение и почему аугментация его уменьшает? б) Почему свертки подходят для мел-спектрограммы, хотя это не фотография? в) Что изменится, если обучать и проверять на записях вперемешку, без разбиения по дикторам?

*Ваш ответ:*

### Задание З6. Матрица ошибок: какие команды путаются и почему

Матрицу ошибок строит готовая функция `confusion` из части 4Г. Допишите:
- `recall_per_class(C)` – полнота каждой команды: верные (диагональ) деленные на сумму строки;
- `most_confused(C)` – пара (истинная команда, предсказанная команда) с наибольшим числом ошибок вне диагонали и это число.

In [ ]:
def confusion(true, pred, n):
    C = np.zeros((n, n), dtype=int)
    for t, p in zip(true, pred):
        C[t, p] += 1
    return C

def recall_per_class(C):
    return ...

def most_confused(C):
    off = ...                                       # <-- обнулить диагональ
    i, j = ...
    return int(i), int(j), int(off[i, j])


C_demo = np.array([[18, 1, 0, 1], [2, 15, 3, 0], [0, 7, 12, 1], [0, 0, 1, 19]])
print(recall_per_class(C_demo).round(3).tolist(), most_confused(C_demo), '  ожидается [0.9, 0.75, 0.6, 0.95] (2, 1, 7)')

best = results['с аугментацией']
pred = predict(best, X_te).numpy()
C = confusion(y_te.numpy(), pred, len(COMMANDS))
print('полнота по командам:', dict(zip(COMMANDS, recall_per_class(C).round(3))))
i, j, n = most_confused(C)
print(f'чаще всего «{COMMANDS[i]}» принимают за «{COMMANDS[j]}»: {n} раз')
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(C, cmap='Blues')
ax.set_xticks(range(6), COMMANDS, rotation=40, ha='right'); ax.set_yticks(range(6), COMMANDS)
for a in range(6):
    for b in range(6):
        ax.text(b, a, C[a, b], ha='center', va='center', color='white' if C[a, b] > C.max() / 2 else NAVY)
ax.set_xlabel('предсказано'); ax.set_ylabel('на самом деле')
show_fig(fig, 'z6_confusion.png')
errors = np.flatnonzero(pred != y_te.numpy())[:3]
for e in errors:
    print(f'на самом деле «{COMMANDS[y_te[e]]}», сеть – «{COMMANDS[pred[e]]}»'); display(Audio(X_wave[~train_mask][e], rate=SR))

**Вопрос З6.** а) Какие команды сеть путает чаще всего? Объясните по звучанию слов (у запасного набора – английских слов) и по их мел-спектрограммам. б) Какую из ошибок опаснее всего допустить в машине? Как это учесть в помощнике, не меняя сеть? в) Послушайте три ошибочные записи: в чем причина – в записи или в сети?

*Ваш ответ:*

### Задания повышенной сложности (к части 1З)

1. **Обнаружение ключевого слова в длинной записи.** Склейте пять записей проверочных дикторов в одну запись длиной 8 секунд, разделив их тишиной со слабым шумом. Двигайте по ней окно в 1 секунду с шагом 0,1 секунды, для каждого положения окна считайте вероятности команд обученной сетью и нарисуйте их график во времени. Команду считаем услышанной, если вероятность выше 0,9 в трех окнах подряд. Сколько команд найдено верно, сколько ложных срабатываний? Что меняется, если добавить седьмой класс «нет команды» (тишина, шум, обрывки слов)?
2. **Устойчивость к шуму двигателя.** Добавьте к проверочным записям шум двигателя (`engine_noise`, обороты 1500 и 3500 в минуту) с отношением сигнал/шум 20, 10, 5 и 0 дБ и постройте график точности двух сетей (без аугментации и с ней) от отношения сигнал/шум. Почему сеть с аугментацией устойчивее и почему при 0 дБ падают обе?
3. **Мел-кепстральные коэффициенты.** Посчитайте MFCC как дискретное косинусное преобразование логарифмической мел-спектрограммы по оси мел-полос (`scipy.fft.dct(..., type=2, norm='ortho', axis=-2)`, первые 13 коэффициентов) и обучите ту же сеть на картинках 13 × 101. Сравните точность и число параметров.

In [ ]:
# Место для заданий повышенной сложности